# Semana 3: Representación del Texto: BOW, TF-IDF y N-gramas
## Notebook de Ejercicios (NB2) – Clasificación de Noticias con BOW/TF-IDF

**Propósito:** Aplicar las técnicas de representación de texto (BOW, TF-IDF, n-gramas) a un problema real de clasificación de noticias y evaluar su rendimiento.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Extraer características de texto usando CountVectorizer y TfidfVectorizer.
- Entrenar un clasificador de regresión logística y evaluar con accuracy.
- Probar distintos rangos de n-gramas y analizar su impacto.
- Identificar las palabras con mayor peso TF-IDF por categoría.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset 20 Newsgroups

El dataset **20 Newsgroups** contiene aproximadamente 20,000 documentos agrupados en 20 categorías. Para simplificar, seleccionaremos 4 categorías relacionadas con ciencia y tecnología.

In [ ]:
# Seleccionamos 4 categorías
categories = [
    'sci.space',
    'sci.med',
    'comp.graphics',
    'rec.sport.baseball'
]

# Cargamos los datos
print("Cargando dataset 20 Newsgroups...")
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"Documentos cargados: {len(newsgroups.data)}")
print(f"Categorías: {newsgroups.target_names}")

# Mostramos un ejemplo
print("\n=== Ejemplo de documento ===")
print(f"Categoría: {newsgroups.target_names[newsgroups.target[0]]}")
print(f"Texto (primeros 500 caracteres):\n{newsgroups.data[0][:500]}...")

### 1.1. Distribución de las categorías

In [ ]:
# Contar documentos por categoría
target_counts = pd.Series(newsgroups.target).value_counts().sort_index()
target_df = pd.DataFrame({
    'Categoría': [newsgroups.target_names[i] for i in target_counts.index],
    'Documentos': target_counts.values
})

print("Distribución de documentos por categoría:")
print(target_df)

plt.figure(figsize=(8, 5))
plt.bar(target_df['Categoría'], target_df['Documentos'])
plt.xlabel('Categoría')
plt.ylabel('Número de documentos')
plt.title('Distribución del dataset 20 Newsgroups')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 2. División en Entrenamiento y Prueba

Dividimos los datos en conjuntos de entrenamiento (80%) y prueba (20%).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=42, stratify=newsgroups.target
)

print(f"Tamaño de entrenamiento: {len(X_train)} documentos")
print(f"Tamaño de prueba: {len(X_test)} documentos")
print(f"\nProporción de clases en entrenamiento:")
for i, cat in enumerate(newsgroups.target_names):
    print(f"  {cat}: {np.mean(y_train == i):.2%}")

---
## 3. Extracción de Características con CountVectorizer (BOW)

Creamos un pipeline que primero vectoriza con CountVectorizer y luego entrena una regresión logística.

In [ ]:
# Pipeline con CountVectorizer
pipeline_count = Pipeline([
    ('vectorizer', CountVectorizer(stop_words='english', max_features=10000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Entrenamos
print("Entrenando con CountVectorizer...")
pipeline_count.fit(X_train, y_train)

# Evaluamos
y_pred_count = pipeline_count.predict(X_test)
accuracy_count = accuracy_score(y_test, y_pred_count)

print(f"\nAccuracy con CountVectorizer: {accuracy_count:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_count, target_names=newsgroups.target_names))

---
## 4. Extracción de Características con TfidfVectorizer

In [ ]:
# Pipeline con TfidfVectorizer
pipeline_tfidf = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', max_features=10000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Entrenamos
print("Entrenando con TfidfVectorizer...")
pipeline_tfidf.fit(X_train, y_train)

# Evaluamos
y_pred_tfidf = pipeline_tfidf.predict(X_test)
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print(f"\nAccuracy con TfidfVectorizer: {accuracy_tfidf:.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_tfidf, target_names=newsgroups.target_names))

### 4.1. Comparación de resultados

In [ ]:
comparacion = pd.DataFrame({
    'Vectorizador': ['CountVectorizer', 'TfidfVectorizer'],
    'Accuracy': [accuracy_count, accuracy_tfidf]
})

print("=== Comparación de Vectorizadores ===")
comparacion

### 4.2. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_count = confusion_matrix(y_test, y_pred_count)
sns.heatmap(cm_count, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=newsgroups.target_names, yticklabels=newsgroups.target_names)
axes[0].set_title('CountVectorizer')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

cm_tfidf = confusion_matrix(y_test, y_pred_tfidf)
sns.heatmap(cm_tfidf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=newsgroups.target_names, yticklabels=newsgroups.target_names)
axes[1].set_title('TfidfVectorizer')
axes[1].set_xlabel('Predicho')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

---
## 5. Experimentación con Diferentes Rangos de N-gramas

Probamos cómo afecta el uso de unigramas, bigramas y trigramas al rendimiento.

In [ ]:
ngram_ranges = [(1,1), (1,2), (1,3), (2,2), (2,3)]
results_ngrams = []

for ngram_range in ngram_ranges:
    pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=ngram_range)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    results_ngrams.append({
        'ngram_range': str(ngram_range),
        'accuracy': accuracy
    })
    print(f"ngram_range={ngram_range}: accuracy={accuracy:.4f}")

df_ngrams = pd.DataFrame(results_ngrams)
print("\n=== Resultados por n-gramas ===")
df_ngrams

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df_ngrams['ngram_range'], df_ngrams['accuracy'])
plt.xlabel('Rango de n-gramas')
plt.ylabel('Accuracy')
plt.title('Impacto del rango de n-gramas en la precisión')
plt.ylim(0.9, 1.0)
plt.show()

---
## 6. Palabras con Mayor Peso TF-IDF por Categoría

Extraemos las palabras con mayor coeficiente en el modelo de regresión logística para cada categoría.

In [ ]:
# Obtenemos el vectorizador y el clasificador del pipeline
vectorizer = pipeline_tfidf.named_steps['vectorizer']
classifier = pipeline_tfidf.named_steps['classifier']

# Nombres de las características (palabras)
feature_names = vectorizer.get_feature_names_out()

# Coeficientes del modelo (para clasificación multiclase, shape: (n_classes, n_features))
coefs = classifier.coef_

# Para cada categoría, mostramos las palabras con mayor coeficiente (más positivas)
print("=== Palabras más importantes por categoría (mayor peso TF-IDF) ===\n")

for i, category in enumerate(newsgroups.target_names):
    # Ordenar coeficientes de mayor a menor
    top_indices = np.argsort(coefs[i])[::-1][:15]
    top_words = feature_names[top_indices]
    top_coefs = coefs[i][top_indices]
    
    print(f"Categoría: {category}")
    for word, coef in zip(top_words, top_coefs):
        print(f"  {word:20} {coef:.4f}")
    print()

### 6.1. Visualización de las palabras más importantes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for i, category in enumerate(newsgroups.target_names):
    top_indices = np.argsort(coefs[i])[::-1][:10]
    top_words = feature_names[top_indices]
    top_coefs = coefs[i][top_indices]
    
    axes[i].barh(top_words, top_coefs)
    axes[i].set_title(f'Categoría: {category}')
    axes[i].set_xlabel('Coeficiente')

plt.tight_layout()
plt.show()

### Análisis de las palabras importantes:

- **sci.space**: palabras como 'space', 'orbit', 'launch', 'nasa', 'satellite' son distintivas.
- **sci.med**: 'medical', 'patient', 'disease', 'treatment', 'doctor'.
- **comp.graphics**: 'graphics', 'image', '3d', 'ray', 'algorithm'.
- **rec.sport.baseball**: 'baseball', 'game', 'team', 'players', 'season'.

Esto confirma que TF-IDF asigna pesos altos a términos relevantes para cada categoría.

---
## 7. Validación Cruzada (Opcional)

Realizamos validación cruzada para obtener una estimación más robusta del rendimiento.

In [ ]:
# Validación cruzada con TfidfVectorizer
from sklearn.model_selection import cross_val_score

pipeline_cv = Pipeline([
    ('vectorizer', TfidfVectorizer(stop_words='english', max_features=10000)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Usamos todo el dataset para CV (puede tomar unos minutos)
print("Realizando validación cruzada (5-fold)...")
scores = cross_val_score(pipeline_cv, newsgroups.data, newsgroups.target, cv=5, scoring='accuracy')

print(f"Accuracy promedio: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")
print(f"Scores individuales: {scores}")

---
## 8. Conclusiones

En este notebook hemos aplicado técnicas de representación de texto a un problema real de clasificación de noticias:

✔️ **CountVectorizer vs TfidfVectorizer**:
- Ambos obtienen resultados muy altos (>95% de accuracy).
- TF-IDF suele dar resultados ligeramente superiores.

✔️ **N-gramas**:
- Incluir bigramas (1,2) mejora el rendimiento.
- Trigramas pueden no aportar mucho y aumentar la dimensionalidad.
- N-gramas mayores (2,2) solos pierden información de unigramas.

✔️ **Palabras importantes por categoría**:
- Las palabras con mayor peso TF-IDF son intuitivamente relevantes.
- Esto ayuda a interpretar el modelo y verificar que aprende patrones sensatos.

**Lección clave**: Las representaciones clásicas (BOW, TF-IDF) combinadas con un clasificador lineal son extremadamente efectivas en clasificación de textos, especialmente cuando se ajustan los n-gramas.

---
**Fin del Notebook de Ejercicios - Semana 3 (NLP)**